### Structured Output

### type OF structured Otput

### 1.Pydantic

Pydantic Models provide the richest feature set with field validation, descriptions and nested structures

In [1]:
import os

from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("qwen/qwen3.6-27b", model_provider="groq")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x10f1db910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f29b1d0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [37]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    actor:str=Field(description="Name of the Actor")
    rating:float=Field(description="The movie rating out of 10")

In [38]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x10f1db910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f29b1d0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'actor': {'description': 'Name of the Actor', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'actor', 'rating'], 'type': 'object'}}}], 'ls_structur

### without using structured output

In [39]:
model.invoke("Provide details about the movie saiyaara")

AIMessage(content='**Saiyaara** is an upcoming Indian romantic drama film, highly anticipated due to its star cast and director. Here are the key details:\n\n### 🎬 Basic Information\n- **Title**: Saiyaara\n- **Director**: Mohit Suri\n- **Producers**: Farhan Akhtar (Raffa Entertainment), Gauri Khan, and Farhad Samji (Excel Entertainment)\n- **Music Composer**: Pritam\n- **Lyricist**: Amitabh (Amitabh Bhattacharya)\n- **Release Date**: **August 22, 2025** (Previously scheduled for 2024, but delayed)\n\n### 🎭 Cast\n- **Ranbir Kapoor** as Arjun\n- **Alia Bhatt** as Pihu\n- **Shraddha Kapoor** as Pihu’s mother (special appearance)\n- **Sajid Faloo** as Arjun’s father\n- **Dolly Ahluwalia** as Arjun’s mother\n- **Bhumi Pednekar** (unconfirmed, but rumored)\n- **Shahana Goswami** (unconfirmed, but rumored)\n\n> *Note: The cast list is based on confirmed reports and official announcements. Some roles may still be under discussion.*\n\n### 📖 Plot Summary\n*Saiyaara* is a romantic drama centered

### With using pydentic structured output

In [40]:
response=model_with_structure.invoke("Provide details about the movie  Dhurandhar")
response

Movie(title='Dhurandhar', year=2025, director='Aditya Dhar', actor='Yash', rating=8.5)

### Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A Movie With details"""
    title : str=Field(..., description="The title of the movie")
    year : int=Field(..., description="This year the movie was released")
    director : str=Field(..., description="The director of the movie")
    rating : float=Field(..., description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Provide details about the movie Inception")
response
    

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8wm3qe0gs', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 355, 'total_tokens': 423, 'completion_time': 0.17134895, 'completion_tokens_details': None, 'prompt_time': 0.025027249, 'prompt_tokens_details': None, 'queue_time': 0.05001377, 'total_time': 0.196376199}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fce7-6fac-7400-9eaf-0a625f9d1805-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': '8wm3qe0gs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 68, 'total_t

### Nested Structure

In [48]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in INR")    

model_with_structure =model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide details about the movie Dhurandhar")  
response

MovieDetails(title='Dhurandhar', year=2025, cast=[Actor(name='Hrithik Roshan', role='Major Vihaan Sharma'), Actor(name='Yash', role='Captain Yash Thakur'), Actor(name='Sai Pallavi', role='Dr. Meera Iyer'), Actor(name='Vaishnavi Madhusudhanan', role='Naina'), Actor(name='Sanjay Dutt', role='Colonel Dev Sharma'), Actor(name='Abhishek Bachchan', role='Major Vikram Sethi')], genres=['Action', 'Thriller', 'Drama', 'War'], budget=3500000000.0)

### 2.TypedDict

TypedDict that provide a simpler alternative, using Python built in typing, ideal when you don't need runtime validation

In [49]:

from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A Movie Or series With details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]

model_with_typedDict_structure = model.with_structured_output(MovieDict)
response = model_with_typedDict_structure.invoke("Please provide the details of the series money Hiest")
response



{'director': 'Álex Pina', 'rating': 8.2, 'title': 'Money Heist', 'year': 2017}

### nested typeddict

In [50]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: Annotated[float | None, ..., "Budget in millions USD"]


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie avenger")
response

{'budget': 220,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Dr. Bruce Banner / Hulk'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'The Avengers',
 'year': 2012}

In [54]:
model.profile

### DataClass


Aa data class is class typically containing mainly data, although they aren't really any restrictions. You create it using the @data class decorator.

### with the help of Pydentic

In [55]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class contactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = init_chat_model("qwen/qwen3.6-27b", model_provider="groq"),
    response_format = contactInfo  # auto-selects ProvideStrategy
)    

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from:John Doe, john@example.com,(555) 123-4567"}]
})
result

{'messages': [HumanMessage(content='Extract contact info from:John Doe, john@example.com,(555) 123-4567', additional_kwargs={}, response_metadata={}, id='112adb81-1579-4177-9efb-d08b6c8e3d3a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'cyaz62s4j', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'contactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 350, 'total_tokens': 415, 'completion_time': 0.213508233, 'completion_tokens_details': None, 'prompt_time': 0.024544098, 'prompt_tokens_details': None, 'queue_time': 0.05104343, 'total_time': 0.238052331}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fcfa-c4a5-7dc0-a242-633e66f798cd-0', tool_calls=[{'name': 'contactInfo', 'args': {'email': 'john@exampl

In [56]:
result["structured_response"]

contactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

### with the help of typedDict


In [57]:
### Using TypeDIct
from typing_extensions import TypedDict,Annotated
from langchain.agents import create_agent

class contactInfo(TypedDict):
    """Contact information for a person."""
    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str, ..., "The phone number of the person"]

agent = create_agent(
    model = init_chat_model("qwen/qwen3.6-27b", model_provider="groq"),
    response_format = contactInfo  # auto-selects ProvideStrategy
)    

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from:John Doe, john@example.com,(555) 123-4567"}]
})
result["structured_response"]



{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

### with the help of dataclass


In [58]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact Information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model = init_chat_model("qwen/qwen3.6-27b", model_provider="groq"),
    response_format = contactInfo  # auto-selects ProvideStrategy
)
result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from:John Doe, john@example.com,(555) 123-4567"}]
})
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}